### Install Modules

In [1]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -qU transformers accelerate
!pip install -qU bitsandbytes pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 20.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 147.1 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.


In [2]:
import os
os.environ["JUPYTER_WIDGETS_DISABLED"] = "true"

In [3]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

PyTorch: 2.8.0+cu126
CUDA available: True
CUDA version: 12.6


### Intialize Embedding Generator

In [4]:
import torch,os,gc
from transformers import AutoTokenizer, AutoModel,T5EncoderModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class EmbeddingGenerator:
    def __init__(self,model_name:str="microsoft/codebert-base", chunk_size:int=128, stride:int=68):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(device)
        self.chunk_size = chunk_size
        self.stride = stride
        if self.tokenizer.pad_token is None:
          self.tokenizer.pad_token = self.tokenizer.eos_token

    def clean_memory(self):
        del self.model,self.tokenizer
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    def generate_embedding(self,code:str):
        all_tokens = self.tokenizer.encode(code, add_special_tokens=False)
        total_length = len(all_tokens)

        if total_length <= self.chunk_size:
            inputs = self.tokenizer(code,return_tensors='pt')
        else:
            chunks = []
            for i in range(0,total_length,self.stride):
                chunk = all_tokens[i:i+self.chunk_size]
                chunks.append(chunk)
            input_ids = [self.tokenizer.build_inputs_with_special_tokens(chunk) for chunk in chunks]
            max_len = max(len(ids) for ids in input_ids)
            attention_masks = []
            padded_input_ids = []
            for chunk in input_ids:
                padding_length = max_len - len(chunk)
                padded_input_ids.append(chunk+[self.tokenizer.pad_token_id]*padding_length)
                attention_masks.append([1]*len(chunk)+[0]*padding_length)
            inputs = {'input_ids': torch.tensor(padded_input_ids), 'attention_mask': torch.tensor(attention_masks)}


        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = self.model(**inputs)

        expanded_attention_mask = inputs['attention_mask'].unsqueeze(-1).expand(outputs.last_hidden_state.shape)
        masked_embeddings = expanded_attention_mask * outputs.last_hidden_state
        code_embedding = (masked_embeddings.sum(1)/ expanded_attention_mask.sum(1)).mean(0)



        if total_length <= self.chunk_size:
            del inputs, outputs, masked_embeddings
        else:
            del inputs,input_ids, attention_masks, outputs, masked_embeddings
        torch.cuda.empty_cache()

        return code_embedding

In [5]:
import ast

class DocstringRemover(ast.NodeTransformer):
    """
    An AST node transformer that removes docstrings from Function, Class,
    and Module nodes.
    """
    def _remove_docstring(self, node):
        """A helper to remove the docstring from a node's body."""
        if not node.body:
            return

        # Check if the first statement is a docstring
        if isinstance(node.body[0], ast.Expr) and isinstance(node.body[0].value, ast.Constant):
            # In Python < 3.8, it's ast.Str, but ast.Constant is used for 3.8+
            # This check is sufficient for modern Python versions.
            node.body = node.body[1:]

    def visit_FunctionDef(self, node):
        self._remove_docstring(node)
        self.generic_visit(node) # Visit children nodes
        return node

    def visit_AsyncFunctionDef(self, node):
        self._remove_docstring(node)
        self.generic_visit(node)
        return node

    def visit_ClassDef(self, node):
        self._remove_docstring(node)
        self.generic_visit(node)
        return node

    def visit_Module(self, node):
        self._remove_docstring(node)
        self.generic_visit(node)
        return node

def remove_all_comments(source_code: str) -> str:
    """
    Removes all comments and docstrings from a Python source code string.

    This function leverages the Abstract Syntax Tree (AST) to safely parse
    and rebuild the code without its comments. It correctly handles multi-line
    strings and other complex cases where regex-based solutions might fail.

    Args:
        source_code: A string containing the Python code.

    Returns:
        The Python code with all comments and docstrings removed.
        Returns the original code if it contains a syntax error.

    Requires: Python 3.9+ for ast.unparse()
    """
    try:
        # 1. Parse the source code into an AST.
        #    Hash comments (#) are automatically discarded during this phase.
        tree = ast.parse(source_code)

        # 2. Traverse the tree to remove docstrings.
        remover = DocstringRemover()
        remover.visit(tree)

        # 3. Unparse the modified AST back into source code.
        return ast.unparse(tree)
    except (SyntaxError, TypeError):
        # If the code has a syntax error, parsing will fail.
        print("Could not parse the source code due to a syntax error.")
        return source_code

In [6]:
# @title
import os

def load_code_from_file(file_path:str)->str:
    with open(file_path,"r") as file:
        code = file.read()
        code = remove_all_comments(code)
    return code

def get_all_python_files(repo_path):
    python_files = []
    for root, _, files in os.walk(repo_path):
        for file in files:
            if file.endswith(".py"):
                python_files.append(os.path.join(root, file))
    return python_files

def get_all_py_files(directory):
    py_files = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith(".py"):
                py_files.append(os.path.join(root, file))
    return py_files

def get_folders(repo_path):
    directories = os.walk(repo_path)
    directories = [i[1] for i in directories][0]
    return directories

### Generate Embeddings

In [7]:
# @title
import pandas as pd
repo_path = "./result/repo_callgraph_clusters"
cluster_embedding_path = "./result/embeddings/embeddings_call_graph_clusters.csv"

embedding_generator = EmbeddingGenerator(model_name="FacebookAI/roberta-base")
embedding_size = embedding_generator.model.config.hidden_size

columns = ["cluster_file"]+[f'dim_{i}' for i in range(embedding_size)]

if os.path.exists(cluster_embedding_path):
  embeddings_df = pd.read_csv(cluster_embedding_path)
else:
  embeddings_df = pd.DataFrame(columns=columns)

python_files = get_all_python_files(repo_path)
for file in python_files:
    code = load_code_from_file(file)
    embeddings = embedding_generator.generate_embedding(code)
    print("Computed embeddings for file:", file)
    embeddings_df.loc[len(embeddings_df)] = [file]+embeddings.tolist()
embedding_generator.clean_memory()
os.makedirs(os.path.dirname(cluster_embedding_path),exist_ok=True)
embeddings_df.to_csv(cluster_embedding_path,index=False)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at FacebookAI/roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
